In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_prefix", "elina_sharabura")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_PREFIX = dbutils.widgets.get("schema_prefix")

BRONZE_SCHEMA = f"{CATALOG}.{SCHEMA_PREFIX}_bronze"
RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA_PREFIX}/raw_data"
CHECKPOINT_BASE = f"/Volumes/{CATALOG}/{SCHEMA_PREFIX}/checkpoints/lab4"

In [0]:
import logging

logger = logging.getLogger("lab4.bronze_ingestion")
logger.setLevel(logging.INFO)
logger.propagate = False
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s - %(message)s"))
    logger.addHandler(handler)

In [0]:
def ingest_to_bronze(table_name, file_pattern):
    table_checkpoint = f"{CHECKPOINT_BASE}/{table_name}"
    (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"{table_checkpoint}/_schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .option("pathGlobFilter", file_pattern)
        .load(RAW_PATH)
        .select(
            "*",
            F.col("_metadata.file_name").alias("source_filename"),
            F.current_timestamp().alias("ingestion_timestamp"),
            F.current_date().alias("load_date"),
        )
        .writeStream
        .option("checkpointLocation", f"{table_checkpoint}/_checkpoint")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"{BRONZE_SCHEMA}.{table_name}")
        .awaitTermination()
    )
    logger.info("%s: ingestion completed", table_name)

In [0]:
ingest_to_bronze("customers", "customers*.csv")
ingest_to_bronze("products", "products*.csv")